<a href="https://colab.research.google.com/github/BariscanTosyali/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BariscanTosyali/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Starter dataset loaded successfully. Shape:", df.shape)

Starter dataset loaded successfully. Shape: (30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Framing Rationale: Content refresh optimization is fundamentally a ranking and scoring problem rather than simple binary classification. Editorial and content engineering teams operate under strict weekly bandwidth constraints; they cannot review thousands of flagged pages all at once. Instead of asking a binary question ("Is this page declining? Yes/No"), the business requires a prioritized output: "Order all candidate pages by their operational urgency for refresh intervention so content creators can execute from top to bottom." Thus, we frame this as predicting a continuous decay probability score to generate an actionable ranked queue.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Variable / Proxy: Binary Organic Decay Indicator (is_declining) derived from recent performance trends.Proxy Definition: True business outcomes (such as permanent revenue loss or missed conversions) are unobservable directly in search logs. Therefore, we construct a ground-truth proxy target derived from performance telemetry:is_declining_label = 1 if trend_direction == "down", else 0.Leakage Prevention: To avoid data leakage, explicit metrics that directly calculate the label—such as trend_pct (the exact percentage drop)—and internal product decision flags are strictly excluded from the feature matrix ($X$). The model is trained purely on observable pre-decision telemetry features (e.g., search volume, impressions, CTR, position, content age, word count).

In [15]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

total = len(df)
declining = df["is_declining_label"].sum()
print(f"Total Pages: {total}")
print(f"Declining Pages (Label = 1): {declining} ({declining/total*100:.2f}%)")

Total Pages: 30000
Declining Pages (Label = 1): 16262 (54.21%)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Success Metric: Precision@K (specifically Precision@50)Defense & Justification: In an operational ranking system, overall accuracy or ROC-AUC are misleading because editorial teams will never review all 30,000 pages. They only have the capacity to act on the top $K$ recommendations (e.g., top 50 pages per week). Precision@50 measures: "Of the top 50 pages our model flags for refresh, what fraction is genuinely declining?"Business Impact: High Precision@50 guarantees that editorial resources are not wasted on false alarms (healthy pages).Target Benchmark: A hand-written rule baseline achieves ~24% Precision@50 (~12/50 correct). Our target ML model must reach $\ge$ 70% Precision@50 (~35/50 correct), representing a $3\times$ operational efficiency gain.

In [16]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

sample_scores = np.random.rand(len(df))
y_true = df["is_declining_label"].values
print(f"Sample Random Baseline Precision@50: {precision_at_k(sample_scores, y_true, 50):.3f}")

Sample Random Baseline Precision@50: 0.620


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique content page (URL / content_id) aggregated over a 90-day historical search telemetry window.

Below is the explicit dataframe slice representing our unit of analysis, showing feature vectors alongside the ground-truth target label:

In [17]:
features_and_target = [
    "content_id", "content_age_days", "days_since_last_update",
    "impressions_90d", "avg_position", "ctr", "word_count", "is_declining_label"
]

unit_df = df[features_and_target].head(5)
print(f"Unit of analysis shape: 1 row = 1 page ({df.shape[0]} total pages)")
unit_df

Unit of analysis shape: 1 row = 1 page (30000 total pages)


,content_id,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,content_304f48230142,187,20,3803,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,445,25,15320,20.3,0.05,2481.0,1
2,content_9aa793d4d895,141,20,12581,36.5,0.09,3515.0,1
3,content_331d6c4de07b,463,22,11751,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,263,14,19140,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why Machine Learning Beats a Fixed Rule:

Non-Linear Feature Interactions: Simple hand-written rules (e.g., days_since_last_update > 180 AND impressions > 500) use rigid thresholds that miss subtle decay. A page updated 120 days ago with a rapid drop from position 3 to 9 is in critical decay, but fixed age rules completely ignore it.

Context-Aware Ranking: Machine learning models (like Random Forests or Gradient Boosted Trees) learn non-linear decision boundaries between position tiers, CTR expectations, and impression volumes simultaneously.

Handling High Tie Rates: Simple rules produce massive "blocks" of tied scores where thousands of pages get the exact same score. ML output probabilities provide a smooth, continuous scoring spectrum that ranks every single URL uniquely for the editorial queue.

In [18]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

hand_p50 = precision_at_k(df["hand_rule_score"], y_true, 50)
print(f"Hand-Written Rule Precision@50: {hand_p50:.3f}")
print(f"Unique Scores Generated by Hand Rule: {df['hand_rule_score'].nunique()} (causes high tie blocks)")

Hand-Written Rule Precision@50: 0.680
Unique Scores Generated by Hand Rule: 18 (causes high tie blocks)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.